# 🧊 Reasoning Segmentation for Sea-Ice SAR — Colab Training

**Indirect-instruction segmentation.** You give the model an *implicit* query that
names an ice type only by its **properties** — e.g. *“segment the land ice that
formed from compressed snow over centuries and flows downhill”* → **glaciers** —
and the model must reason what is meant and segment the matching ice (or **nothing**
if that ice is not in the image). This is the sea-ice analogue of *“segment the fruit
with the most vitamin C” → the orange.*

**How the text is made to matter (contrastive queries).** Each training sample is
randomly either:
- **positive** — the indirect query matches the image's ice type → target = the raw `_scat` mask, or
- **negative** — the query describes a *different* ice type → target = **empty** (segment nothing).

So the model can't just segment the salient blob; it has to reason whether the
*described* ice is present. Query bank: `data/reasoning_queries.py`. Toggle with
`cfg.data.reasoning_seg_mode`.

**Honest metrics:** `pos_mIoU` (segment the right ice on matching queries),
`neg_reject` (stay empty on non-matching queries), and `reasoning_score` (their mean —
the number to optimise and report). Classification is **image-only** (`cls_image_only=True`)
so its F1 can't leak from the text.

> ⚠️ Runtime → **Change runtime type → GPU** (A100/L4 recommended; T4 auto-falls back to fp16).

## 1. 🖥️ Check GPU

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'No GPU found — switch the runtime type to GPU!')

## 2. 📂 Mount Google Drive (optional — persistent checkpoints)

In [ ]:
USE_DRIVE = True  # set False to keep everything in the ephemeral VM
DRIVE_DIR = '/content/drive/MyDrive/reasoning_seg'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import os; os.makedirs(DRIVE_DIR, exist_ok=True)
    print('Checkpoints will be copied to', DRIVE_DIR)

## 3. 📥 Clone the repository

In [ ]:
import os
REPO_URL = 'https://github.com/prakhar443/Reasoning_Segmentation_new.git'
BRANCH   = 'claude/charming-cerf-2rh1i7'  # reasoning-segmentation branch

if not os.path.exists('/content/Reasoning_Segmentation_new'):
    !git clone --branch {BRANCH} {REPO_URL} /content/Reasoning_Segmentation_new
%cd /content/Reasoning_Segmentation_new
!git log --oneline -3

## 4. 📦 Install dependencies (~3 min on first run)

In [ ]:
# Install project deps
!pip install -q -r requirements.txt

# ── Colab compat fix ──────────────────────────────────────────────────────
# Colab ships torchao 0.10.0, but the installed peft's LoRA dispatcher rejects
# any torchao < 0.16.0 (raises ImportError during get_peft_model). This
# pipeline uses plain LoRA on CLIP — no torchao quantization — so removing
# torchao makes peft skip that code path cleanly.
!pip uninstall -q -y torchao 2>/dev/null || true

import torch, albumentations, transformers, peft
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('albumentations', albumentations.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__)
try:
    import torchao  # noqa: F401
    print('WARNING: torchao still present:', torchao.__version__)
except ImportError:
    print('torchao removed (OK) — peft will skip the torchao LoRA path')

## 5. 🔎 Sanity check — new GT + text channel

Verify the three pillars of the redesign before burning GPU hours:
the soft scat ground truth, the per-image descriptions, and the text-guided decoder flag.

In [ ]:
from config import cfg
print('reasoning_seg_mode    :', cfg.data.reasoning_seg_mode, '  (indirect query -> mask)')
print('negative_ratio        :', cfg.data.reasoning_negative_ratio)
print('mask_target_mode      :', cfg.data.mask_target_mode, '   (soft_scat = raw scat maps are GT)')
print('cls_image_only        :', cfg.model.cls_image_only, '   (honest, text-free classification)')
print('text_guided_decoder   :', cfg.model.text_guided_decoder)

from data.dataset import SeaIceDataset
ds = SeaIceDataset('dataset', split='test', use_augmentation=False)
pos = next(s for s in (ds[i] for i in range(len(ds))) if float(s['is_positive']) >= 0.5)
neg = next(s for s in (ds[i] for i in range(len(ds))) if float(s['is_positive']) <  0.5)
print('\n--- POSITIVE example (query matches the image) ---')
print('query :', pos['short_desc'])
print('target: non-empty scat mask, max =', round(float(pos['mask'].max()), 3))
print('\n--- NEGATIVE example (query describes a different ice) ---')
print('query :', neg['short_desc'])
print('target: empty mask, max =', round(float(neg['mask'].max()), 3), '(model must segment nothing)')

## 6. 📉 Degenerate floor under the new ground truth

All-foreground baseline against the **scat** GT — the number the trained model must clear on mIoU.

In [ ]:
!python baseline_allforeground.py --split test --gt scat

## 7. 🚀 Train

Defaults in `config.py` are the reasoning-segmentation configuration. Early stopping on val mIoU; best-mIoU and best-F1 checkpoints are saved separately.

*A100/L4*: leave `BATCH_SIZE=4`. *T4 (16 GB)*: use `BATCH_SIZE=2`.

In [ ]:
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ''
BATCH_SIZE = 2 if 'T4' in gpu else 4
EPOCHS = 60
print(f'GPU: {gpu} → batch_size={BATCH_SIZE}, epochs={EPOCHS}')

!python train.py --output_dir outputs --batch_size {BATCH_SIZE} --epochs {EPOCHS}

## 8. 💾 Back up checkpoints to Drive

In [ ]:
if USE_DRIVE:
    !cp -v outputs/best_model.pth outputs/best_model_f1.pth {DRIVE_DIR}/ 2>/dev/null || true

## 9. 📊 Evaluate on the held-out test split

Metrics are computed against the **scat ground truth** (soft GT binarised at 0.5 of its normalised range).

In [ ]:
!python evaluate.py --checkpoint outputs/best_model.pth --output eval_results/ --save_visualizations --save_masks

## 10. 🏁 Compare against the previous model

In [ ]:
import json
rep = json.load(open('eval_results/test_evaluation_report.json'))

print('REASONING-SEGMENTATION (the headline numbers):')
if 'reasoning_score' in rep:
    print(f"  pos mIoU   (segment the referred ice) : {rep['pos_miou']:.3f}")
    print(f"  neg reject (empty on wrong query)     : {rep['neg_reject']:.3f}")
    print(f"  reasoning_score (balanced)            : {rep['reasoning_score']:.3f}")
else:
    print('  (reasoning metrics absent — reasoning_seg_mode was off)')

PREV = {'mean_iou': 0.351, 'weighted_f1': 0.778, 'accuracy': 0.833}
print('\nVS PREVIOUS MODEL:')
print(f"{'Metric':<30}{'Previous':>10}{'This run':>10}")
print('-'*50)
print(f"{'Segmentation mIoU (plain)':<30}{PREV['mean_iou']:>10.3f}{rep['mean_iou']:>10.3f}")
print(f"{'Classification accuracy':<30}{PREV['accuracy']:>10.3f}{rep['accuracy']:>10.3f}")
print(f"{'Classification weighted F1':<30}{PREV['weighted_f1']:>10.3f}{rep['weighted_f1']:>10.3f}")
print('\nNote: plain mIoU is inflated by empty/empty=1 on negative queries —')
print('use pos_mIoU + neg_reject (reasoning_score) as the real segmentation result.')

## 11. 🖼️ Qualitative results — text-navigated masks

Image · soft scat GT · prediction, with the description that guided each mask.

In [ ]:
import torch, textwrap
import matplotlib.pyplot as plt
from config import cfg
from data.dataset import SeaIceDataset, collate_fn
from models.pipeline import SeaIceSegmentationPipeline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ckpt = torch.load('outputs/best_model.pth', map_location=device)
model = SeaIceSegmentationPipeline(cfg.model, use_sam=False).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

ds = SeaIceDataset('dataset', split='test', use_augmentation=False)
idxs = [0, 15, 30, 45, 60, 75]   # two per a few classes
fig, axes = plt.subplots(len(idxs), 3, figsize=(12, 3.2 * len(idxs)))
for r, i in enumerate(idxs):
    b = collate_fn([ds[i]])
    with torch.no_grad():
        out = model(images=b['image'].to(device), descriptions=b['long_desc'])
    img = b['image'][0].permute(1, 2, 0).cpu().numpy()
    gt  = b['mask'][0, 0].cpu().numpy()
    pr  = out['masks_gated'][0, 0].cpu().numpy()
    axes[r, 0].imshow(img); axes[r, 0].set_title('SAR image', fontsize=9)
    axes[r, 1].imshow(gt, cmap='viridis', vmin=0, vmax=1); axes[r, 1].set_title('GT: raw scat map', fontsize=9)
    axes[r, 2].imshow(pr, cmap='viridis', vmin=0, vmax=1); axes[r, 2].set_title('Prediction', fontsize=9)
    for ax in axes[r]: ax.axis('off')
    axes[r, 0].text(0, -18, textwrap.fill('“' + b['long_desc'][0][:140] + '…”', 110), fontsize=7)
plt.tight_layout(); plt.savefig('eval_results/qualitative.png', dpi=120); plt.show()

## 12. 🧭 Reasoning demo — indirect query controls the mask

Same image, different **indirect** instructions. A matching property-description should
segment the ice; a non-matching one should come back (almost) **empty**. This is the
behaviour the old constant-prompt model could not exhibit.

In [ ]:
import textwrap, torch
import matplotlib.pyplot as plt
from data.reasoning_queries import REASONING_QUERIES

sample = ds.samples[0]
true_cls = sample['ice_class']
other_cls = next(c for c in REASONING_QUERIES if c != true_cls)
queries = [
    (f'MATCHING ({true_cls})',     REASONING_QUERIES[true_cls][0]),
    (f'NON-MATCH ({other_cls})',   REASONING_QUERIES[other_cls][0]),
]
b = collate_fn([ds[0]])
fig, axes = plt.subplots(1, len(queries)+1, figsize=(4*(len(queries)+1), 4))
axes[0].imshow(b['image'][0].permute(1,2,0).cpu().numpy())
axes[0].set_title(f'SAR image\n(true: {true_cls})', fontsize=9); axes[0].axis('off')
for c,(tag,q) in enumerate(queries):
    with torch.no_grad():
        out = model(images=b['image'].to(device), descriptions=[q])
    pres = float(out['presence_prob'][0])         # reasoning: is it present?
    pr = out['masks_gated'][0,0].cpu().numpy()     # gated production mask
    axes[c+1].imshow(pr, cmap='viridis', vmin=0, vmax=1)
    axes[c+1].set_title(f'{tag}\nP(present)={pres:.2f}', fontsize=8)
    axes[c+1].set_xlabel(textwrap.fill(q[:90], 34), fontsize=6)
    axes[c+1].set_xticks([]); axes[c+1].set_yticks([])
plt.tight_layout(); plt.show()
print('Reasoning works if: MATCHING has high P(present) + a mask;')
print('NON-MATCH has low P(present) + an empty (gated) mask.')

## 13. 📝 Notes — squeezing out more performance

- **More epochs**: bump `EPOCHS` in section 7; early stopping (patience 12 on val mIoU) prevents wasted compute.
- **Unredacted text**: `cfg.data.scrub_class_names = False` lets full descriptions through — segmentation usually gains a little and classification F1 rises sharply, but the F1 is then no longer leakage-free. Keep the default for honest numbers.
- **Wider decoder**: `cfg.model.decoder_base_channels = 48` if you are on an A100.
- **Legacy comparison**: set `cfg.data.mask_target_mode = 'binary'` to reproduce the old Otsu-GT training.
- The best-F1 checkpoint is saved separately as `outputs/best_model_f1.pth`.